# Exploratory Data Analysis

This notebook uses simple plots to understand the Raisin Dataset and decide whether scaling and degree-two features are reasonable.

In [ ]:
from pathlib import Path

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="colorblind",
)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 120,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.titlesize": 13,
})

## 1. Dataset Loading

The unchanged Raisin Dataset is loaded from the project's raw-data directory.

In [ ]:
CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

DATA_PATH = (PROJECT_ROOT/"data"/"raw"/"raisin"/"Raisin_Dataset.xlsx")


if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

FIGURE_OUTPUT_DIRECTORY = (
    PROJECT_ROOT / "reports" / "figures" / "eda"
)
FIGURE_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

def save_figure(figure, file_name):
    figure.savefig(
        FIGURE_OUTPUT_DIRECTORY / f"{file_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURE_OUTPUT_DIRECTORY / f"{file_name}.pdf",
        bbox_inches="tight",
    )

In [ ]:
df = pd.read_excel(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

## 2. Class Distribution

The class distribution is examined to determine whether the target variable is balanced.

In [ ]:
TARGET_COLUMN = "Class"
CLASS_ORDER = ["Kecimen","Besni"]

CLASS_COLORS = {
    "Kecimen": "#0072B2",
    "Besni": "#D55E00",
}

class_counts = (df[TARGET_COLUMN].value_counts().reindex(CLASS_ORDER))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

sns.countplot(
    data=df,
    x=TARGET_COLUMN,
    order=CLASS_ORDER,
    hue=TARGET_COLUMN,
    palette=CLASS_COLORS,
    legend=False,
    ax=ax,
)

for bar in ax.patches:
    count = int(bar.get_height())
    percentage = count / len(df)

    ax.annotate(
        f"{count}\n({percentage:.1%})",
        xy=(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
        ),
        xytext=(0, 6),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax.set_title("Class Distribution of the Raisin Dataset")
ax.set_xlabel("Raisin Variety")
ax.set_ylabel("Number of Samples")
ax.set_ylim(0, class_counts.max() * 1.18)

sns.despine()
plt.tight_layout()
save_figure(fig, "class_distribution")
plt.show()

### Interpretation

The classes are balanced: 450 Kecimen and 450 Besni samples.

## 3. Numerical Feature Distributions

Histograms are used to examine the shape, spread, skewness, and class overlap of each numerical feature.

In [ ]:
numerical_features = (
    df.select_dtypes(include=np.number)
    .columns
    .tolist()
)


print("Numerical features:")
print(numerical_features)

In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(15, 11),
)

axes = axes.flatten()

for index, feature in enumerate(numerical_features):
    ax = axes[index]

    sns.histplot(
        data=df,
        x=feature,
        hue=TARGET_COLUMN,
        hue_order=CLASS_ORDER,
        palette=CLASS_COLORS,
        bins=25,
        kde=True,
        stat="density",
        common_norm=False,
        element="step",
        alpha=0.25,
        legend=(index == 0),
        ax=ax,
    )

    ax.set_title(feature)
    ax.set_xlabel("")
    ax.set_ylabel("Density")

for index in range(len(numerical_features), len(axes)):
    axes[index].set_visible(False)

fig.suptitle(
    "Numerical Feature Distributions by Raisin Variety",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()
save_figure(fig, "numerical_feature_histograms")
plt.show()

### Interpretation

Size-related features are right-skewed. Their class distributions differ, but no single feature cleanly separates the classes.

## 4. Class-Wise Box Plots

Box plots are used to compare the location, spread, and potential outliers of each numerical feature across the two raisin varieties.

In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(15, 12),
)

axes = axes.flatten()

for index, feature in enumerate(numerical_features):
    ax = axes[index]

    sns.boxplot(
        data=df,
        x=TARGET_COLUMN,
        y=feature,
        order=CLASS_ORDER,
        hue=TARGET_COLUMN,
        palette=CLASS_COLORS,
        legend=False,
        width=0.6,
        showfliers=True,
        flierprops={
            "marker": "o",
            "markerfacecolor": "none",
            "markeredgecolor": "#444444",
            "markersize": 4,
            "alpha": 0.6,
        },
        ax=ax,
    )

    ax.set_title(feature)
    ax.set_xlabel("")
    ax.set_ylabel("Value")

for index in range(len(numerical_features), len(axes)):
    axes[index].set_visible(False)

fig.suptitle(
    "Numerical Features by Raisin Variety",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()
save_figure(fig, "class_wise_boxplots")
plt.show()

### Interpretation

Besni generally has larger size measurements. The plots also show possible outliers, but a plot alone is not a reason to delete them.

## 5. Correlation Matrix

The Pearson correlation matrix is used to examine linear relationships and potential redundancy among the numerical features.

In [ ]:
correlation_matrix = (
    df[numerical_features]
    .corr(method="pearson")
)

correlation_matrix

In [ ]:
upper_triangle_mask = np.triu(
    np.ones_like(correlation_matrix, dtype=bool),
    k=1,
)

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    mask=upper_triangle_mask,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.6,
    cbar_kws={
        "label": "Pearson Correlation",
        "shrink": 0.8,
    },
    ax=ax,
)

ax.set_title(
    "Correlation Matrix of Numerical Features",
    pad=16,
)

plt.tight_layout()
save_figure(fig, "correlation_matrix")
plt.show()

### Interpretation

Area and ConvexArea are almost perfectly correlated, and Perimeter is strongly related to other size features. Model coefficients should therefore be interpreted carefully.

## 6. Class-Colored Scatter Plots

Selected feature pairs are examined to identify class separation, overlap,
curved relationships, and unusual observations.

In [ ]:
selected_feature_pairs = [
    ("MajorAxisLength", "MinorAxisLength"),
    ("Eccentricity", "Perimeter"),
    ("Area", "Extent"),
]

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(18, 5.5),
)

for index, (x_feature, y_feature) in enumerate(
    selected_feature_pairs
):
    ax = axes[index]

    sns.scatterplot(
        data=df,
        x=x_feature,
        y=y_feature,
        hue=TARGET_COLUMN,
        hue_order=CLASS_ORDER,
        palette=CLASS_COLORS,
        alpha=0.65,
        s=45,
        edgecolor="white",
        linewidth=0.3,
        legend=(index == 0),
        ax=ax,
    )

    correlation_value = correlation_matrix.loc[
        x_feature,
        y_feature,
    ]

    ax.set_title(
        f"{y_feature} vs {x_feature}\n"
        f"Pearson r = {correlation_value:.2f}"
    )

    ax.set_xlabel(x_feature)
    ax.set_ylabel(y_feature)

fig.suptitle(
    "Selected Feature Relationships by Raisin Variety",
    fontsize=16,
    fontweight="bold",
    y=1.03,
)

plt.tight_layout()
save_figure(fig, "selected_feature_scatterplots")
plt.show()

### Interpretation

The classes overlap, and some relationships are curved. This motivates one simple comparison with degree-two square and interaction features.

## 7. Outlier Investigation

The IQR rule is used to identify potential outliers. These observations are
reviewed as candidates rather than removed automatically.

In [ ]:
outlier_flags = pd.DataFrame(
    False,
    index=df.index,
    columns=numerical_features,
)

outlier_summary_rows = []

for feature in numerical_features:
    first_quartile = df[feature].quantile(0.25)
    third_quartile = df[feature].quantile(0.75)

    iqr = third_quartile - first_quartile

    lower_bound = first_quartile - 1.5 * iqr
    upper_bound = third_quartile + 1.5 * iqr

    feature_outlier_flags = (
        (df[feature] < lower_bound)
        | (df[feature] > upper_bound)
    )

    outlier_flags[feature] = feature_outlier_flags

    candidate_count = feature_outlier_flags.sum()

    outlier_summary_rows.append(
        {
            "Feature": feature,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Candidate Count": candidate_count,
            "Candidate Percentage": (
                candidate_count / len(df) * 100
            ),
        }
    )

outlier_summary = pd.DataFrame(outlier_summary_rows)

outlier_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

sns.barplot(
    data=outlier_summary,
    x="Candidate Count",
    y="Feature",
    color="#0072B2",
    ax=ax,
)

ax.bar_label(
    ax.containers[0],
    padding=4,
    fontsize=9,
)

ax.set_title("Potential Outlier Count by Feature")
ax.set_xlabel("Number of IQR Outlier Candidates")
ax.set_ylabel("")

sns.despine()
plt.tight_layout()
save_figure(fig, "outlier_candidate_counts")
plt.show()

### Interpretation

The IQR rule flags several observations. They are candidates for inspection, not confirmed errors.

In [ ]:
validity_checks = pd.Series(
    {
        "Non-positive Area": (
            df["Area"] <= 0
        ).sum(),

        "Non-positive ConvexArea": (
            df["ConvexArea"] <= 0
        ).sum(),

        "Non-positive axis length": (
            (df["MajorAxisLength"] <= 0)
            | (df["MinorAxisLength"] <= 0)
        ).sum(),

        "Non-positive Perimeter": (
            df["Perimeter"] <= 0
        ).sum(),

        "Eccentricity outside [0, 1]": (
            ~df["Eccentricity"].between(0, 1)
        ).sum(),

        "Extent outside [0, 1]": (
            ~df["Extent"].between(0, 1)
        ).sum(),

        "Major axis shorter than minor axis": (
            df["MajorAxisLength"]
            < df["MinorAxisLength"]
        ).sum(),

        "ConvexArea smaller than Area": (
            df["ConvexArea"]
            < df["Area"]
        ).sum(),
    },
    name="Violation Count",
)

validity_checks.to_frame()

In [ ]:
outlier_count_per_row = outlier_flags.sum(axis=1)

outlier_feature_names = outlier_flags.apply(
    lambda row: ", ".join(row.index[row]),
    axis=1,
)

outlier_review = df.loc[
    outlier_count_per_row > 0
].copy()

outlier_review["Outlier Feature Count"] = (
    outlier_count_per_row[outlier_count_per_row > 0]
)

outlier_review["Outlier Features"] = (
    outlier_feature_names[outlier_count_per_row > 0]
)

outlier_review = outlier_review.sort_values(
    by="Outlier Feature Count",
    ascending=False,
)

outlier_review.head(20)

### Outlier Decision

No rows are removed because the flagged values remain physically plausible.

## 8. Class Balance Assessment

The target classes are examined using the documented binary-label convention:
Kecimen is the negative class (`0`) and Besni is the positive class (`1`).

In [ ]:
NEGATIVE_CLASS = "Kecimen"
POSITIVE_CLASS = "Besni"

CLASS_TO_BINARY = {
    NEGATIVE_CLASS: 0,
    POSITIVE_CLASS: 1,
}

In [ ]:
class_balance_counts = (df[TARGET_COLUMN].value_counts().reindex([NEGATIVE_CLASS, POSITIVE_CLASS]))
class_balance_proportions = (
    class_balance_counts
    / class_balance_counts.sum()
)
class_balance_summary = pd.DataFrame(
    {
        "Binary Label": [
            CLASS_TO_BINARY[class_name]
            for class_name in class_balance_counts.index
        ],
        "Count": class_balance_counts,
        "Proportion": class_balance_proportions,
        "Percentage": (
            class_balance_proportions * 100
        ),
    }
)

class_balance_summary.index.name = "Class"

class_balance_summary


In [ ]:
majority_count = class_balance_counts.max()
minority_count = class_balance_counts.min()

imbalance_ratio = (
    majority_count / minority_count
)

print(f"Majority-class count: {majority_count}")
print(f"Minority-class count: {minority_count}")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")

### Interpretation

The negative and positive classes both contain 450 samples, giving a 1:1 ratio.

## 9. Accuracy and Class Imbalance

Accuracy measures the proportion of correctly classified observations.
However, it can be misleading when one class is substantially more common
than the other.

In [ ]:
example_negative_count = 950
example_positive_count = 50

example_total_count = (
    example_negative_count
    + example_positive_count
)

always_negative_correct_predictions = (
    example_negative_count
)

example_accuracy = (
    always_negative_correct_predictions
    / example_total_count
)

correct_positive_predictions = 0

example_positive_recall = (
    correct_positive_predictions
    / example_positive_count
)

print(f"Accuracy: {example_accuracy:.1%}")
print(f"Positive-class recall: {example_positive_recall:.1%}")

### Metric Decision

Accuracy is meaningful because the classes are balanced. Precision, recall, F1-score, and the confusion matrix will still be reported to show both types of error.

## 10. Stratified Train–Validation–Test Split

The dataset is divided into training, validation, and test sets using
stratification. A 60/20/20 split is selected so that each subset preserves the
original 50/50 class distribution.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

train_df, temporary_df = train_test_split(
    df,
    test_size=0.40,
    random_state=RANDOM_SEED,
    stratify=df[TARGET_COLUMN],
)


In [ ]:
from sklearn.model_selection import train_test_split

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temporary_df[TARGET_COLUMN],
)


In [ ]:
print(f"Training shape: {train_df.shape}")
print(f"Validation shape: {validation_df.shape}")
print(f"Test shape: {test_df.shape}")

In [ ]:
def split_summary(split_name, split_data):
    counts = (
        split_data[TARGET_COLUMN]
        .value_counts()
        .reindex([NEGATIVE_CLASS, POSITIVE_CLASS])
    )

    percentages = (
        counts
        / counts.sum()
        * 100
    )

    return pd.DataFrame(
        {
            "Split": split_name,
            "Class": counts.index,
            "Count": counts.values,
            "Percentage": percentages.values,
        }
    )

In [ ]:
split_distribution_summary = pd.concat(
    [
        split_summary(
            "Training",
            train_df,
        ),
        split_summary(
            "Validation",
            validation_df,
        ),
        split_summary(
            "Test",
            test_df,
        ),
    ],
    ignore_index=True,
)

split_distribution_summary

In [ ]:
train_indices = set(train_df.index)
validation_indices = set(validation_df.index)
test_indices = set(test_df.index)

assert train_indices.isdisjoint(validation_indices)
assert train_indices.isdisjoint(test_indices)
assert validation_indices.isdisjoint(test_indices)

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == len(df)
)

print("All data splits are mutually exclusive.")
print("All observations are accounted for.")

In [ ]:
APPLY_RESAMPLING = False

if APPLY_RESAMPLING:
    raise NotImplementedError(
        "Resampling must be implemented only for the training set."
    )
else:
    resampled_train_df = train_df.copy()

print("Resampling applied:", APPLY_RESAMPLING)
print("Training observations:", len(resampled_train_df))